In [3]:
import os
from google import genai
from google.genai import types

from tools import lookup_order, calculate

In [ ]:
os.environ["GOOGLE_API_KEY"] 

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [5]:
lookup_order_tool = types.FunctionDeclaration(
    name="lookup_order",
    description="Look up an order by order ID and return item, price, purchase date, warranty.",
    parameters={
        "type": "object",
        "properties": {
            "order_id": {"type": "string"}
        },
        "required": ["order_id"]
    }
)

calculate_tool = types.FunctionDeclaration(
    name="calculate",
    description="Evaluate arithmetic expressions like 1200 * 3",
    parameters={
        "type": "object",
        "properties": {
            "expression": {"type": "string"}
        },
        "required": ["expression"]
    }
)

tools = [
    types.Tool(
        function_declarations=[
            lookup_order_tool,
            calculate_tool
        ]
    )
]

In [6]:
def run_tool(name, args):
    if name == "lookup_order":
        return lookup_order(args["order_id"])

    if name == "calculate":
        return calculate(args["expression"])

    raise ValueError(f"Unknown tool: {name}")

In [10]:
question = "For order A1001, what would the total be if I bought three of them?"
contents = [question]

while True:

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=contents,
        config=types.GenerateContentConfig(tools=tools)
    )

    part = response.candidates[0].content.parts[0]

    # ---------------------------
    # FINAL ANSWER
    # ---------------------------
    if not hasattr(part, "function_call") or part.function_call is None:
        print("\nFINAL ANSWER:\n")
        print(response.text)
        break

    # ---------------------------
    # TOOL CALL
    # ---------------------------
    fc = part.function_call
    name = fc.name
    args = dict(fc.args)

    print("\nTOOL CALL DETECTED")
    print("Tool:", name)
    print("Args:", args)

    # ---------------------------
    # RUN TOOL
    # ---------------------------
    result = run_tool(name, args)

    print("TOOL RESULT:", result)

    # ---------------------------
    # FEED BACK TO MODEL
    # ---------------------------
    contents.append(response.candidates[0].content)

    contents.append(
        types.Content(
            role="tool",
            parts=[
                types.Part.from_function_response(
                    name=name,
                    response={"result": result}
                )
            ]
        )
    )


TOOL CALL DETECTED
Tool: lookup_order
Args: {'order_id': 'A1001'}
TOOL RESULT: {'item': 'laptop', 'price': 1200, 'purchased': '2026-05-20', 'warranty_months': 12}

TOOL CALL DETECTED
Tool: calculate
Args: {'expression': '1200 * 3'}
TOOL RESULT: 3600

FINAL ANSWER:

The price of one laptop (order A1001) is $1,200. Therefore, the total for three of them would be $3,600.


In [11]:
question = "What can you help me with?"
contents = [question]

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=contents,
    config=types.GenerateContentConfig(tools=tools)
)

print("\nFINAL ANSWER:\n")
print(response.text)


FINAL ANSWER:

I can assist you with a variety of tasks, ranging from information retrieval to managing specific account details. Here are a few things I can do for you:

*   **Order Tracking:** If you have an order ID, I can look up the details for you, including the items purchased, price, purchase date, and warranty information.
*   **Calculations:** I can perform arithmetic calculations (like 1200 * 3) if you need help with numbers.
*   **Answering Questions:** I can provide information, explain concepts, or help you brainstorm ideas.
*   **Writing & Editing:** I can help draft emails, write essays, create summaries, or refine text you've already written.

**How can I help you today?** If you have an order ID you'd like me to check or a question you'd like me to answer, feel free to ask!


In [15]:
#Invalid order test (A9999)

question = "Find order A9999 and multiply its price by 2"
contents = [question]

while True:

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=contents,
        config=types.GenerateContentConfig(tools=tools)
    )

    part = response.candidates[0].content.parts[0]

    if not hasattr(part, "function_call") or part.function_call is None:
        print("\nFINAL ANSWER:\n")
        print(response.text)
        break

    fc = part.function_call
    name = fc.name
    args = dict(fc.args)

    print("\nTOOL CALL:", name, args)

    result = run_tool(name, args)

    print("TOOL RESULT:", result)

    contents.append(response.candidates[0].content)

    contents.append(
        types.Content(
            role="tool",
            parts=[
                types.Part.from_function_response(
                    name=name,
                    response={"result": result}
                )
            ]
        )
    )


TOOL CALL: lookup_order {'order_id': 'A9999'}
TOOL RESULT: {'error': 'Order A9999 not found'}

FINAL ANSWER:

The order A9999 could not be found. Please double-check the order ID and try again.
